In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from gen_variable_standard_static import metrics_search_for_two_fragments_df
from clean_and_collect_power_data_alt import Clean
from tqdm import tqdm
import pyarrow as pa

In [31]:
for package_choice in [np, pd, pa]:
    print(package_choice.__name__)
    print(package_choice.__version__)

numpy
2.3.5
pandas
3.0.0
pyarrow
23.0.0


In [2]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
test_path = '../../../../data_ds_project/testing_yearly_parquet/'

In [3]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [4]:
ac_power_metrics = metrics_search_for_two_fragments_df(metrics_df, 'ac', 'pow', 'and')
ac_power_systems = set(ac_power_metrics['system_id'])
two_years_days = 2 * 365
enough_data_systems = systems_cleaned[systems_cleaned['num_days_actual_records'] >= two_years_days]
enough_data_ids = set(enough_data_systems.system_id)
enough_data_parquet_power_systems = enough_data_ids.intersection(ac_power_systems)
enough_data_parquet_power_list = list(enough_data_parquet_power_systems)
enough_data_parquet_power_list.sort()

In [5]:
test_path = '../../../../data_ds_project/testing_yearly_parquet/'
# amend as necessary

For testing, just save locally (don't upload it, I think)

In [18]:
test_10 = Clean(10, test_path, systems_cleaned, None, './test_results_10_first/')

In [19]:
my_year = 2016
year_2016_data_basic = test_10.extract_years_data_parquet(my_year)
year_2016_data_standard = test_10.standardize_dataframe(year_2016_data_basic)
year_2016_data_rem = test_10.remove_small_values(year_2016_data_standard)
year_2016_good_days = test_10.keep_good_days_only(year_2016_data_rem)
year_2016_energy_readings = test_10.convert_to_energy_LRS(year_2016_good_days)

In [19]:
test_1200 = Clean(1200, test_path, systems_cleaned, 'inverter', './test_results_1200_i_renew/')

In [ ]:
def part_by_part(clean_obj: Clean, year: int):
    try:
        year_data_basic = clean_obj.extract_years_data_parquet(year)
    except BaseException as e:
        print(f'Trouble with {clean_obj.system_id}, part {clean_obj.meter_or_inverter}, year {year}.')
        print(f'Trouble with extract_years_data_parquet')
        raise e
    try:
        year_data_standard = clean_obj.standardize_dataframe(year_data_basic)
    except BaseException as e:
        print(f'Trouble with {clean_obj.system_id}, part {clean_obj.meter_or_inverter}, year {year}.')
        print(f'Trouble with standardize_dataframe')
        raise e
    try:
        year_data_rem = clean_obj.remove_small_values(year_data_standard)
    except BaseException as e:
        print(f'Trouble with {clean_obj.system_id}, part {clean_obj.meter_or_inverter}, year {year}.')
        print(f'Trouble with remove_small_values')
        raise e
    try:
        year_data_good_days = clean_obj.keep_good_days_only(year_data_rem)
    except BaseException as e:
        print(f'Trouble with {clean_obj.system_id}, part {clean_obj.meter_or_inverter}, year {year}.')
        print(f'Trouble with keep_good_days_only (probably good_days).')
        raise e
    try:
        year_data_energy_readings = clean_obj.convert_to_energy_LRS(year_data_good_days)
    except BaseException as e:
        print(f'Trouble with {clean_obj.system_id}, part {clean_obj.meter_or_inverter}, year {year}.')
        print(f'Trouble with convert_to_energy_LRS')
        raise e
    return (year_data_basic, year_data_standard, year_data_rem, year_data_good_days, year_data_energy_readings)

In [20]:
test_1200_year_2012 = gen_loop(test_1200, 2012)

In [21]:
data_1200_inv_bas, data_1200_inv_std, data_1200_inv_rem, data_1200_inv_good, data_1200_year_ener = test_1200_year_2012

In [23]:
data_1200_inv_bas['ac_power_kW_inverter'].max()

np.float64(48.6627193)

In [24]:
data_1200_inv_std

,time,power
1,2012-01-01 07:20:31,0.000000
2,2012-01-01 07:25:32,0.000210
3,2012-01-01 07:30:32,0.017430
4,2012-01-01 07:35:32,0.106918
5,2012-01-01 07:40:32,0.203117
...,...,...
34004,2012-12-13 16:45:32,0.024229
34005,2012-12-13 16:50:32,0.000000
34006,2012-12-13 16:55:32,0.000000
34007,2012-12-13 17:00:32,0.000000


In [25]:
data_1200_inv_rem

,time,power
7,2012-01-01 07:50:32,0.590158
8,2012-01-01 07:55:32,1.115535
9,2012-01-01 08:00:32,1.692868
10,2012-01-01 08:05:32,2.239591
11,2012-01-01 08:10:33,2.888722
...,...,...
33998,2012-12-13 16:15:31,1.936600
33999,2012-12-13 16:20:31,1.618567
34000,2012-12-13 16:25:31,1.291200
34001,2012-12-13 16:30:31,0.940933


In [18]:
systems_cleaned[systems_cleaned['system_id']==1200]

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year
11,1200,Distributed Sun - BWI Hilton,"Linthicum Heights, MD",America/New_York,39.1958,-76.6808,155.0,51.84,Cfa,34,...,True,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,3365,2011


In [16]:
data_1200_inv_std

,time,power


Quick loop

In [5]:
for system_id in tqdm(enough_data_parquet_power_list):
    # grab columns from a blank year
    blank_year_pq = pq.ParquetDataset(
        f'{test_path}{system_id}/',
        filters = [('year', '==', 1990)]
    )
    blank_year_df = blank_year_pq.read().to_pandas()
    pow_cols = [col_name for col_name in blank_year_df.columns if ('pow' in col_name)]
    is_inv = any([('inv' in col_name) for col_name in pow_cols])
    is_met = any([('met' in col_name) for col_name in pow_cols])
    if is_inv and is_met:
        met_inv_inputs = ['inverter', 'meter']
        met_inv_names = met_inv_inputs
    elif is_inv:
        met_inv_inputs = ['inverter',]
        met_inv_names = met_inv_inputs
    elif is_met:
        met_inv_inputs = ['meter',]
        met_inv_names = met_inv_inputs
    else:
        met_inv_inputs = [None,]
        met_inv_names = ['unknown',]
    for j in range(len(met_inv_inputs)):
        met_or_inv = met_inv_inputs[j]
        met_or_inv_name = met_inv_names[j]
        system_cleaning_module = Clean(system_id, test_path, systems_cleaned,
                                       met_or_inv, 
                                       f'./test_results/{system_id}/{met_or_inv_name}/')
        system_cleaning_module.clean_all_and_write_to_file()


  0%|          | 0/82 [00:00<?, ?it/s]

100%|██████████| 82/82 [02:30<00:00,  1.83s/it]
